# BrandSphere AI — Week 7: Smart Social & Brand Campaign Studio
**CRS AI Capstone 2025-26 | Scenario 1**

This notebook:
1. Downloads Marketing & Startups datasets from Kaggle
2. Cleans and encodes features (platform, region, campaign type)
3. Trains Random Forest + Linear Regression to predict CTR, ROI, Engagement
4. Validates with RMSE and feature importance
5. Saves models for Streamlit integration

## Cell 1 — Install Dependencies

In [ ]:
!pip install kagglehub pandas scikit-learn matplotlib seaborn plotly numpy -q
print('✅ Dependencies installed')

## Cell 2 — Download Marketing & Startups Datasets

In [ ]:
import kagglehub, os, pandas as pd, numpy as np

# Marketing dataset
print('📥 Downloading Marketing Dataset...')
try:
    mkt_path = kagglehub.dataset_download('manishabhatt22/marketing-campaign-performance-dataset')
    print(f'✅ Marketing dataset: {mkt_path}')
except:
    mkt_path = None
    print('⚠️  Using synthetic marketing data')

# Load or generate marketing data
if mkt_path:
    for root, dirs, files in os.walk(mkt_path):
        for f in files:
            if f.endswith('.csv'):
                df_mkt = pd.read_csv(os.path.join(root, f))
                print(f'✅ Loaded: {f} — {df_mkt.shape}')
                print(df_mkt.head(3))
                break
else:
    # Generate synthetic marketing dataset
    np.random.seed(42)
    n = 2000
    platforms    = np.random.choice(['Instagram','Facebook','Twitter/X'], n)
    regions      = np.random.choice(['North America','Europe','Asia Pacific','Middle East','Latin America'], n)
    objectives   = np.random.choice(['Brand Awareness','Engagement','Conversion'], n)
    personalities= np.random.choice(['minimalist','vibrant','luxury','bold','elegant'], n)
    budget       = np.random.uniform(500, 10000, n)

    base_ctr = {'Instagram':3.2,'Facebook':1.8,'Twitter/X':2.1}
    base_roi = {'Brand Awareness':120,'Engagement':210,'Conversion':310}
    base_eng = {'Instagram':65,'Facebook':48,'Twitter/X':55}
    pers_b   = {'vibrant':1.2,'bold':1.15,'minimalist':0.95,'luxury':1.1,'elegant':1.05}

    ctr = np.array([base_ctr[p] * pers_b[pp] + np.random.normal(0, 0.3) for p, pp in zip(platforms, personalities)])
    roi = np.array([base_roi[o] * pers_b[pp] + np.random.normal(0, 15) for o, pp in zip(objectives, personalities)])
    eng = np.array([base_eng[p] * pers_b[pp] + np.random.normal(0, 5) for p, pp in zip(platforms, personalities)])
    ctr = np.clip(ctr, 0.5, 8.0)
    roi = np.clip(roi, 50, 500)
    eng = np.clip(eng, 20, 95)

    df_mkt = pd.DataFrame({'platform': platforms, 'region': regions,
                           'objective': objectives, 'personality': personalities,
                           'budget': budget, 'ctr': ctr, 'roi': roi, 'engagement': eng})
    print(f'✅ Synthetic marketing dataset generated: {df_mkt.shape}')
    print(df_mkt.head())

## Cell 3 — Data Inspection & Quality Check

In [ ]:
print('📊 Dataset Info:')
print(df_mkt.info())
print('\n📊 Missing values:')
print(df_mkt.isnull().sum())
print('\n📊 Descriptive statistics:')
print(df_mkt.describe())

## Cell 4 — Data Cleaning & Feature Engineering

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Identify target columns dynamically
df = df_mkt.copy().dropna()

# Standardize column names to lowercase
df.columns = df.columns.str.lower().str.replace(' ', '_')
print('Columns:', df.columns.tolist())

# Identify categorical columns to encode
cat_cols = ['platform', 'region', 'objective', 'personality',
            'campaign_type', 'channel', 'target_audience']
cat_cols = [c for c in cat_cols if c in df.columns]

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col+'_enc'] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f'✅ Encoded {col}: {le.classes_}')

# Define feature columns
feature_cols = [c+'_enc' for c in cat_cols]
if 'budget' in df.columns:
    feature_cols.append('budget')
if 'ad_spend' in df.columns:
    feature_cols.append('ad_spend')

print(f'\n✅ Feature columns: {feature_cols}')

## Cell 5 — Train Models to Predict CTR, ROI, Engagement

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Identify target columns dynamically
target_map = {}
for possible in [('ctr','click_through_rate','ctr_percent'), ('roi','return_on_investment'), ('engagement','engagement_score','engagement_rate')]:
    for col in possible:
        if col in df.columns:
            target_map[possible[0]] = col
            break

print(f'✅ Targets found: {target_map}')

X = df[feature_cols].fillna(0)
models = {}
results = {}

for target_name, target_col in target_map.items():
    y = df[target_col].fillna(df[target_col].median())
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train Random Forest
    rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    models[target_name] = rf
    results[target_name] = {'rmse': rmse, 'r2': r2, 'model': 'RandomForest'}
    print(f'\n🎯 {target_name.upper()} — Random Forest')
    print(f'   RMSE: {rmse:.4f}')
    print(f'   R²:   {r2:.4f}')

## Cell 6 — Feature Importance Plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, len(models), figsize=(6*len(models), 5))
fig.suptitle('Random Forest Feature Importance for KPI Prediction', fontsize=14, fontweight='bold')

if len(models) == 1:
    axes = [axes]

for ax, (target, model) in zip(axes, models.items()):
    importances = model.feature_importances_
    feat_names  = [c.replace('_enc','') for c in feature_cols]
    sorted_idx  = np.argsort(importances)[::-1]
    ax.bar([feat_names[i] for i in sorted_idx],
           [importances[i] for i in sorted_idx],
           color='#C9A84C')
    ax.set_title(f'{target.upper()} Predictors', fontsize=11)
    ax.set_xlabel('Feature')
    ax.set_ylabel('Importance')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature importance chart saved')

## Cell 7 — Predicted vs Actual Plot

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(6*len(models), 5))
fig.suptitle('Predicted vs Actual KPI Values', fontsize=14, fontweight='bold')

if len(models) == 1:
    axes = [axes]

for ax, (target, model) in zip(axes, models.items()):
    target_col = target_map[target]
    y = df[target_col].fillna(df[target_col].median())
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    y_pred = model.predict(X_te)
    ax.scatter(y_te, y_pred, alpha=0.4, color='#3ECFB2', s=20)
    mn, mx = min(y_te.min(), y_pred.min()), max(y_te.max(), y_pred.max())
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect fit')
    ax.set_xlabel(f'Actual {target.upper()}')
    ax.set_ylabel(f'Predicted {target.upper()}')
    ax.set_title(f'{target.upper()} — R²={results[target]["r2"]:.3f}')
    ax.legend()

plt.tight_layout()
plt.savefig('predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Predicted vs Actual plot saved')

## Cell 8 — Regional Engagement Analysis

In [ ]:
if 'region' in df.columns and 'engagement' in target_map:
    eng_col = target_map['engagement']
    regional = df.groupby('region')[eng_col].mean().sort_values(ascending=False)
    plt.figure(figsize=(10, 5))
    bars = plt.bar(regional.index, regional.values, color='#1B3A6B')
    plt.title('Average Engagement Score by Region', fontsize=14, fontweight='bold')
    plt.xlabel('Region')
    plt.ylabel('Avg Engagement Score')
    plt.xticks(rotation=20)
    for bar, val in zip(bars, regional.values):
        plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{val:.1f}', ha='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('regional_engagement.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Regional engagement chart saved')
else:
    print('Region/engagement column not found — skipping')

## Cell 9 — Save Models & Upload to Drive

In [ ]:
import pickle, shutil
from google.colab import drive

# Save all models
with open('campaign_models.pkl', 'wb') as f:
    pickle.dump({'models': models, 'encoders': encoders,
                 'feature_cols': feature_cols, 'target_map': target_map,
                 'results': results}, f)

print('✅ campaign_models.pkl saved')

drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/BrandSphere_Models'
os.makedirs(save_dir, exist_ok=True)

for fname in ['campaign_models.pkl', 'feature_importance.png',
              'predicted_vs_actual.png', 'regional_engagement.png']:
    if os.path.exists(fname):
        shutil.copy(fname, os.path.join(save_dir, fname))
        print(f'   ✅ Uploaded {fname}')

## Cell 10 — Summary

In [ ]:
print('=' * 60)
print('  BRANDSPHERE AI — CAMPAIGN MODEL SUMMARY')
print('=' * 60)
print(f'  Dataset:    Marketing Campaign Performance')
print(f'  Records:    {len(df)}')
print(f'  Features:   {feature_cols}')
print(f'  Algorithm:  Random Forest Regressor (100 trees)')
print()
for target, res in results.items():
    print(f'  {target.upper():<12} RMSE: {res["rmse"]:.4f}  |  R²: {res["r2"]:.4f}')
print('=' * 60)
print('  OUTPUTS:')
print('  - campaign_models.pkl      → trained models')
print('  - feature_importance.png   → model insights')
print('  - predicted_vs_actual.png  → validation plot')
print('  - regional_engagement.png  → geographic insights')
print('=' * 60)